# Notebook 04: Camera Basics

## ADAS Connection
Every self-driving car starts with **perception** -- the ability to see and understand the world around it. Tesla, Waymo, and every other autonomous vehicle company has spent billions of dollars on this one problem: how do you teach a computer to see?

It all starts with a camera and a single frame of video. In this notebook you will see exactly what your robot sees, and learn how a computer interprets an image -- not as a picture, but as a grid of numbers.

---

## How It Works
A digital image is not a picture -- it is a **grid of pixels**. Each pixel has three numbers representing its color: Red, Green, and Blue (RGB). A 640x480 image contains 307,200 pixels, each with 3 values -- that is nearly 1 million numbers just to describe one frame of video.

Self-driving cars process **30 or more frames per second** -- which means the computer is crunching hundreds of millions of numbers every single second just to see the road.

OpenCV is the library that handles this for us. It captures frames from the camera and gives us the pixel data as a **NumPy array** -- a grid of numbers we can inspect, modify, and analyze.

---

## The Code
Run this cell first to set up the camera and display tools.

In [ ]:
import cv2
import numpy as np
import ipywidgets as widgets
import threading
import time
from IPython.display import display

def bgr8_to_jpeg(frame):
    return bytes(cv2.imencode('.jpg', frame)[1])

# Open the camera
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH,  640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
cap.set(cv2.CAP_PROP_FPS, 30)
cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter.fourcc('M','J','P','G'))

ret, test_frame = cap.read()
if ret:
    print(f'Camera ready!')
    print(f'Frame size: {test_frame.shape[1]} x {test_frame.shape[0]} pixels')
    print(f'Total pixels per frame: {test_frame.shape[0] * test_frame.shape[1]:,}')
    print(f'Total values per frame: {test_frame.size:,} (each pixel has 3 color values)')
else:
    print('ERROR: Could not open camera. Check that /dev/video0 exists.')

---

## YOUR TURN -- Tweak Zone 1: Look at a Single Pixel

Change the X and Y coordinates below to inspect different pixels in the frame.
Each pixel has three values: Blue, Green, Red (OpenCV uses BGR order, not RGB).

- Try a pixel in a bright area vs a dark area
- Try a pixel on something red vs something blue

> **Think like an engineer:** Why do you think OpenCV uses BGR instead of RGB? What would happen if you mixed up the order?

In [ ]:
# ═══════════════════════════════════════
#   TWEAK THESE VALUES
PIXEL_X = 320   # horizontal position (0 to 639)
PIXEL_Y = 240   # vertical position (0 to 479)
# ═══════════════════════════════════════

ret, frame = cap.read()
if ret:
    pixel = frame[PIXEL_Y, PIXEL_X]
    print(f'Pixel at ({PIXEL_X}, {PIXEL_Y}):')
    print(f'  Blue:  {pixel[0]}')
    print(f'  Green: {pixel[1]}')
    print(f'  Red:   {pixel[2]}')
    print(f'  Raw values: {pixel}')
    
    # Draw a circle on the frame to show which pixel we are looking at
    cv2.circle(frame, (PIXEL_X, PIXEL_Y), 10, (0, 255, 255), 2)
    
    image_widget = widgets.Image(format='jpeg', width=640, height=480)
    display(image_widget)
    image_widget.value = bgr8_to_jpeg(frame)

---

## YOUR TURN -- Tweak Zone 2: Live Camera Feed

Now let's look at a live feed. Change the display settings and observe what happens.

Run the START cell to begin the feed, then run the STOP cell when you are done.

> **Think like an engineer:** Notice the FPS counter in the top left. Self-driving cars need at least 30 FPS to react in time. What happens to FPS as you increase the resolution?

In [ ]:
# ═══════════════════════════════════════
#   TWEAK THESE VALUES
DISPLAY_WIDTH  = 640    # try 320, 480, 640
DISPLAY_HEIGHT = 480    # try 240, 360, 480
SHOW_FPS       = True   # True or False
# ═══════════════════════════════════════

feed_widget = widgets.Image(format='jpeg', width=DISPLAY_WIDTH, height=DISPLAY_HEIGHT)
display(feed_widget)

running = True
fps_start = time.time()
fps_count = 0

def camera_feed():
    global running, fps_count, fps_start
    while running:
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.resize(frame, (DISPLAY_WIDTH, DISPLAY_HEIGHT))
        if SHOW_FPS:
            fps_count += 1
            fps = fps_count / (time.time() - fps_start)
            cv2.putText(frame, f'FPS: {fps:.1f}', (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
        feed_widget.value = bgr8_to_jpeg(frame)
        time.sleep(0.033)

feed_thread = threading.Thread(target=camera_feed)
feed_thread.daemon = True
feed_thread.start()
print('Feed started. Run the STOP cell when done.')

In [ ]:
# STOP CELL -- run this to stop the feed
running = False
time.sleep(0.5)
print('Feed stopped.')

---

## YOUR TURN -- Tweak Zone 3: Color Spaces

OpenCV can convert an image between different **color spaces**. The most important one for us is **HSV (Hue, Saturation, Value)** -- it separates color (hue) from brightness (value), which makes it much easier to detect a specific color under different lighting conditions.

This is why self-driving cars use HSV for color detection -- a red stop sign looks different in bright sunlight vs shade in RGB, but stays similar in HSV.

> **Think like an engineer:** Why would lighting conditions be a problem for a self-driving car trying to read a stop sign? What time of day would be hardest?

In [ ]:
# ═══════════════════════════════════════
#   TWEAK THIS VALUE
COLOR_SPACE = 'HSV'   # try 'BGR', 'HSV', 'GRAY'
# ═══════════════════════════════════════

ret, frame = cap.read()
if ret:
    if COLOR_SPACE == 'HSV':
        converted = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        title = 'HSV Color Space'
    elif COLOR_SPACE == 'GRAY':
        converted = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        converted = cv2.cvtColor(converted, cv2.COLOR_GRAY2BGR)  # convert back for display
        title = 'Grayscale'
    else:
        converted = frame
        title = 'BGR (original)'
    
    # Show original and converted side by side
    combined = np.hstack([frame, converted])
    cv2.putText(combined, 'Original BGR', (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,255), 2)
    cv2.putText(combined, title, (650, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,255), 2)
    
    compare_widget = widgets.Image(format='jpeg', width=1280, height=480)
    display(compare_widget)
    compare_widget.value = bgr8_to_jpeg(combined)
    print(f'Showing: {title}')

---

## What Happened?

Think about these questions with your team:

1. What did the pixel values tell you about a bright vs dark area of the image?
2. How did the image look different in HSV vs BGR?
3. Why do you think grayscale might be useful for some ADAS applications but not others?
4. A 640x480 camera at 30 FPS generates how many pixel values per second? Calculate it.

---

## CHALLENGE -- Advanced Students

Write code that captures a frame and finds the **brightest pixel** in the image. Print its coordinates and RGB values.

Hint: look up `cv2.minMaxLoc()` and `cv2.cvtColor()` with `cv2.COLOR_BGR2GRAY`.

In [ ]:
# YOUR CODE HERE
ret, frame = cap.read()
if ret:
    # Convert to grayscale to find brightness
    # Find the brightest pixel
    # Print its location and color values
    pass

---

## Always clean up when you are done!

In [ ]:
running = False
time.sleep(0.5)
cap.release()
print('Camera released.')